# Proyek Akhir: Menyelesaikan Permasalahan Perusahaan Edutech

- Nama: Zulfi Sam Shiddiq
- Email: zulfisamshiddi09.03@gmail.com
- Id Dicoding: zulfi_sam_shiddiq

## 1. Persiapan

### 1.1 Menyiapkan library yang dibutuhkan

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from imblearn.over_sampling import SMOTE
import joblib

# Menonaktifkan peringatan (warnings) agar notebook terlihat rapi
import warnings
warnings.filterwarnings('ignore')

### 1.2 Menyiapkan data yang akan diguankan

In [ ]:
df = pd.read_csv("data/data.csv", sep=";")
df.head()

## 2. Data Understanding

### 2.1 Mengecek informasi dasar data

In [ ]:
df.info()

In [ ]:
display(df.isnull().sum())
print("\nJumlah Data Duplikat:", df.duplicated().sum())

In [ ]:
df.describe()

Berdsasarkan observasi awal, dataset terdiri dari 4424 rows dan 37 columns. Tidak ditemukan missing values dan duplikasi data. Seluruh feature predictor dalam sudah dalam format numerik dengan kolom target `Status` bertipe object. Dari statistik deskriptif, terlihat adanya perbedaan skala yang signifikan antar feature, misalnya `Course` dengan `Inflation_rate` sehingga perlukan proses scaling

### 2.2 EDA

In [ ]:
# Cek distribusi data
print("Jumlah data per kelas pada kolom Status:")
print(df['Status'].value_counts())

plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='Status', palette='viridis', order=df['Status'].value_counts().index)

for p in ax.patches:
    ax.annotate(format(p.get_height(), '.0f'), 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'center', 
                xytext = (0, 9), 
                textcoords = 'offset points')

plt.title('Distribusi Status Mahasiswa', fontsize=14)
plt.xlabel('Status', fontsize=12)
plt.ylabel('Jumlah Mahasiswa', fontsize=12)
plt.show()

Berdsasarkan observasi awal, dataset terdiri dari 4424 rows dan 37 columns. Tidak ditemukan missing values dan duplikasi data. Seluruh feature predictor dalam sudah dalam format numerik dengan kolom target `Status` bertipe object. Dari statistik deskriptif, terlihat adanya perbedaan skala yang signifikan antar feature, misalnya `Course` dengan `Inflation_rate` sehingga perlukan proses scaling. Kemudian dari distribusi data kolom `Status` memeiliki distribusi data yang tidak normal. Jumalh ketiga data bersifat imbalanced/tidak seimbang

### 2.3 Analisis Korelasi (Heatmap)

In [ ]:
df_eda = df.copy()

df_eda['Status_encoded'] = df_eda['Status'].map({'Dropout': 0, 'Enrolled': 1, 'Graduate': 2})

numeric_cols = df_eda.select_dtypes(include=[np.number])

corr_matrix = numeric_cols.corr()

target_corr = corr_matrix['Status_encoded'].drop('Status_encoded')
top_features = target_corr[abs(target_corr) > 0.2].index.tolist()

plt.figure(figsize=(12, 8))
top_corr_matrix = numeric_cols[top_features + ['Status_encoded']].corr()

mask = np.triu(np.ones_like(top_corr_matrix, dtype=bool))

sns.heatmap(top_corr_matrix, annot=True, mask=mask, cmap='RdBu', fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title('Heatmap Korelasi: Faktor Paling Berpengaruh terhadap Kelulusan Mahasiswa', fontsize=15, pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Secara umum terdapat tiga faktor utama yang memiliki korelasi antar variabel, yaitu<br>
1. Faktor akademik
   - `Curricular_units_2nd_sem_approved` dan ` 1st-sem_approved` dengan jumlah mata kuliah yang lulus sangat menentukkan
   - `Curricular_units_2nd_sem_grade` dan `1st_sem_grade`, nilai yang baik berbanding lulus dengan kelulusan
2. Faktor finansial
   - `Tuition_fees_up_to_date`, menegaskan mahasiswa yang SPP nya lancar ccenderung lulus
   - `Scholarship_holder`, ini juga berbanding lulus bagi mahasiswa yang mendapatkan beasiswa aman dari dropout
   - `Debtor`, mahasiswa yang berstatus debitur/punya tunggakan sangat rentan dropout
3. Faktor demografi
   - `Age_at_enrollment`, semakin tua usia saat mendaftar, akan kecenderungan lebih tinggi untuk dropout
   - `Gender`, korelasi negatif ini menunjukkan bahwa salah satu gender memiliki tingkat dropout yang lebih tinggi, yaitu pria.

### 2.4 Deep Dive Eksplorasi Variabel Utama

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Analisis Mendalam Faktor Utama Terhadap Status Mahasiswa', fontsize=18, fontweight='bold', y=1.02)

# 1. Faktor Akademik (Boxplot: Mata Kuliah yang Lulus di Semester 2)
sns.boxplot(ax=axes[0, 0], data=df, x='Status', y='Curricular_units_2nd_sem_approved', palette='muted')
axes[0, 0].set_title('Distribusi Mata Kuliah Lulus (Semester 2) berdasar Status', fontsize=14)
axes[0, 0].set_xlabel('Status')
axes[0, 0].set_ylabel('Jumlah Mata Kuliah Lulus')

# 2. Faktor Finansial (Countplot: Keteraturan Bayar SPP)
sns.countplot(ax=axes[0, 1], data=df, x='Tuition_fees_up_to_date', hue='Status', palette='viridis')
axes[0, 1].set_title('Pengaruh Kelancaran Bayar SPP terhadap Status', fontsize=14)
axes[0, 1].set_xlabel('Tuition Fees Up to Date (0 = Nunggak, 1 = Lancar)')
axes[0, 1].set_ylabel('Jumlah Mahasiswa')

# 3. Faktor Finansial (Countplot: Status Beasiswa)
sns.countplot(ax=axes[1, 0], data=df, x='Scholarship_holder', hue='Status', palette='Set2')
axes[1, 0].set_title('Pengaruh Status Beasiswa terhadap Status Mahasiswa', fontsize=14)
axes[1, 0].set_xlabel('Scholarship Holder (0 = Tidak, 1 = Ya)')
axes[1, 0].set_ylabel('Jumlah Mahasiswa')

# 4. Faktor Demografi (KDE Plot / Distribusi: Umur Saat Mendaftar)
sns.kdeplot(ax=axes[1, 1], data=df, x='Age_at_enrollment', hue='Status', fill=True, palette='coolwarm', common_norm=False)
axes[1, 1].set_title('Distribusi Umur Mahasiswa Saat Mendaftar berdasar Status', fontsize=14)
axes[1, 1].set_xlabel('Umur Saat Mendaftar')
axes[1, 1].set_ylabel('Density (Kepadatan)')

plt.tight_layout()
plt.show()

1. **Performa akademik :** Terdapat batasan pemisahan yang sangat tajam pada performa mahasiswa di akhir smester 2. Mahasiswa yang berstatus dorpout mayoritas memlkiki jumlah mata kuliah lulus antara 0-4. Sebaliknya, mahasiswa yang graduate memiliki penyelesaian mata kuliah yang sangat bai, mayoritas lulusan 5-7 mata kuliah. Mahasiswa yang gagal mendaatkan SKS penuh di tahun pertama adalah kandidat utama yang berisiko tinggi untuk dropout.
2. **Kepatuhan finansial SPP :** Kelancaran pembayaran UKT pada column `Tuition_fess_up_to_date` adalah indikator yang sangat krusial. Pada cluster mahasisa yang menunggak SPP (0), status dropout mendominasi secrara absolut. Sangat setikit mahasiswa yang menunggak namun bisa lulus. Masalah finansial ini merulaman penyebab dropout tercepat, universitas perlu memberikan peringatan dini atau bantuan konsultasi finansial sejak tunggakan pertama terjadi.
3. **Kepemilikan beasiswa :** Seperti yang terdapat pada kolom `Schoolarship_holder` berperan sebagai pengaman yang sangat kuat. Penerima beasiswa ini memiliki tingkat lulusan yang sangat tinggi dan rasio dropout yang sangat minim dibandinkan dengan yang tidak menerima beasiswa. Program beasiswa ini terbukti efektif mempertahankan mahasiswa hingga lulus.
4. **Usia saat mendaftar :** Distribusi usia menunjukkan bahwa mahasiswa yang lulus sangat terkonsetrasi pada usia mudia sekitar 18-20an, namun pada kurva mahasiswa dropout berada dikisaran usian 30-50am. Dengan usia dewasa atau non reguler ini memiliki tantangan yang lebih besar untuk bertahan seperti harus membagi waktu dengan pekerjaan atau keluarga, sehingga dibutuhkan fleksibilitas atau pembelajaran berdiferensiasi.

## 3. Data Preparation / Preprocessing

### 3.1 Pemisahan data Enrolled untuk Inferensi/Prediksi

In [ ]:
df_enrolled = df[df['Status'] == 'Enrolled'].copy()
df_enrolled.head()

### 3.2 Filter dataset utama (Dropout dan Graduate)

In [ ]:
df_train = df[df['Status'] != 'Enrolled'].copy()
df_train.head()

In [ ]:
print("Dataset berhasil dipisah!")
print(f"Jumlah data training (Dropout & Graduate) : {len(df_train)} baris")
print(f"Jumlah data inferensi (Enrolled)          : {len(df_enrolled)} baris")

### 3.3 Memisahkan X dan y

In [ ]:
X = df_train.drop(columns=['Status'])
y = df_train['Status']

### 3.4 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Jumlah X_train:", len(X_train))
print("Jumlah X_test:", len(X_test))

### 3.5 Scaling (Standarisasi)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nData berhasil di-split dan di-scale!")

### 3.6 Imnbalaced data handling (SMOTE)

In [ ]:
# Cek proporsi sebelum SMOTE
print("Proporsi kelas SEBELUM SMOTE:")
print(y_train.value_counts())

# Terapkan SMOTE pada data training yang sudah di-scale
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Cek proporsi setelah SMOTE
print("\nProporsi kelas SETELAH SMOTE:")
print(y_train_resampled.value_counts())

## 4. Modeling

Alasan menggunakan RFC karena memiliki karakteristik utama sebagai model yang tahan terhadap overfitting dan memiliki kemampuan menangani dataset tabular dengan baik.

### 4.1 Inisialisasi model RandomForest

In [ ]:
# rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model = RandomForestClassifier(random_state=42)

#### 4.2 Model training

In [ ]:
# rf_model.fit(X_train_balanced, y_train_balanced)
rf_model.fit(X_train_resampled, y_train_resampled)

### 4.3 Prediksi

In [ ]:
y_pred = rf_model.predict(X_test_scaled)

## 5. Evaluation

### 5.1 Akurasis model

In [ ]:
print("Akurasi Model:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

### 5.2 Inferensi

In [ ]:
# enyiapkan fitur dari data Enrolled
X_enrolled = df_enrolled.drop(columns=['Status'])

# Lakukan scaling
X_enrolled_scaled = scaler.transform(X_enrolled)

# Memprediksi status masa depan
enrolled_predictions = rf_model.predict(X_enrolled_scaled)

# Memasukkan hasil prediksi ke dataframe Enrolled
df_enrolled['Predicted_Future_Status'] = enrolled_predictions

print("Berdasarkan model ML, nasib mahasiswa yang saat ini 'Enrolled' diprediksi akan menjadi:")
print(df_enrolled['Predicted_Future_Status'].value_counts())

### 5.3 Save model

In [ ]:
joblib.dump(rf_model, 'model/rf_model.joblib')
joblib.dump(scaler, 'model/scaler.joblib')
print("Model dan Scaler terbaru berhasil disimpan dan siap digunakan untuk Streamlit!")

- **Prediksi Graduate :** Berdasarkan nilai recall, model dapat menebak mahasiswa yang lulus dengan kebenaran sebesar 89%
- **Prediksi Dropout :** Berdasarkan nilai precision, model dapat menebak mahasiswa yang dropout dengan akurat sebesar 83%
- **Prediksi Enrolled :** Status enrolled adalah fase transisi/zona abu-abu. Berdasarkan kemungkinan fakta yang terjadi, enrolled memiliki nilai bagus mirip dengan graduate dan memililiki tunggakan SPP/UKT mirip dengan dropout. Sehinggat Machine Learning terjadi kebingungan untuk membedakan kelas ini.

## Ekspor Data Tableau

In [ ]:
# Menambahkan Student_ID ke dataframe asli untuk sinkronisasi dashboard
df_dashboard = df.copy()
df_dashboard.insert(0, 'Student_ID', range(1, 1 + len(df_dashboard)))

# Ekspor ke CSV
df_dashboard.to_csv('data/data_for_dashboard.csv', index=False)
print("File 'data_for_dashboard.csv' berhasil dibuat di folder data/!")